In [1]:
# CELL 1 - Imports and basic setup

import math
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
# CELL 2 - Download Yahoo Finance multi-asset OHLCV data

tickers = [
    "AAPL",     # Apple
    "MSFT",     # Microsoft
    "GOOG",     # Alphabet
    "AMZN",     # Amazon
    "TSLA",     # Tesla
    "NVDA",     # NVIDIA
    "META",     # Meta Platforms
    "SPY",      # S&P 500 ETF
    "GLD",      # Gold ETF
    "BTC-USD",  # Bitcoin (USD)
]

start_date = "2015-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    threads=True,
)

# Ensure consistent OHLCV columns for each ticker
# We'll keep: Open, High, Low, Close, Adj Close, Volume -> 6 features per ticker
features = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

# Build a single DataFrame with 60 columns (10 tickers × 6 features)
all_cols = []
for ticker in tickers:
    for f in features:
        all_cols.append(f"{ticker}_{f}")

df_list = []
for ticker in tickers:
    df_t = data[ticker][features].copy()
    df_t.columns = [f"{ticker}_{c}" for c in df_t.columns]
    df_list.append(df_t)

df = pd.concat(df_list, axis=1)
df = df.dropna()
print("Data shape:", df.shape)
df.head()

[*********************100%***********************]  10 of 10 completed


Data shape: (2764, 60)


,AAPL_Open,AAPL_High,AAPL_Low,AAPL_Close,AAPL_Adj Close,AAPL_Volume,MSFT_Open,MSFT_High,MSFT_Low,MSFT_Close,...,GLD_Low,GLD_Close,GLD_Adj Close,GLD_Volume,BTC-USD_Open,BTC-USD_High,BTC-USD_Low,BTC-USD_Close,BTC-USD_Adj Close,BTC-USD_Volume
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,27.847500,27.860001,26.837500,27.332500,24.237539,212818400.0,46.660000,47.419998,46.540001,46.759998,...,112.320000,114.080002,114.080002,7109600.0,314.079010,315.838989,313.565002,315.032013,315.032013,7860650.0
2015-01-05,27.072500,27.162500,26.352501,26.562500,23.554745,257142000.0,46.369999,46.730000,46.250000,46.330002,...,114.730003,115.800003,115.800003,8177400.0,265.084015,278.341003,265.084015,274.473999,274.473999,43962800.0
2015-01-06,26.635000,26.857500,26.157499,26.565001,23.556955,263188400.0,46.380001,46.750000,45.540001,45.650002,...,115.800003,117.120003,117.120003,11238300.0,274.610992,287.553009,272.696014,286.188995,286.188995,23245700.0
2015-01-07,26.799999,27.049999,26.674999,26.937500,23.887280,160423600.0,45.980000,46.459999,45.490002,46.230000,...,116.169998,116.430000,116.430000,6434200.0,286.076996,298.753998,283.079010,294.337006,294.337006,24866800.0
2015-01-08,27.307501,28.037500,27.174999,27.972500,24.805080,237458000.0,46.750000,47.750000,46.720001,47.590000,...,115.849998,115.940002,115.940002,7033700.0,294.135010,294.135010,282.174988,283.348999,283.348999,19982500.0


In [3]:
# CELL 3 - Train/val/test split (80/10/10) and normalization

values = df.values.astype(np.float32)

n_total = len(values)
n_train = int(n_total * 0.8)
n_val = int(n_total * 0.1)
n_test = n_total - n_train - n_val

train_data = values[:n_train]
val_data = values[n_train:n_train + n_val]
test_data = values[n_train + n_val:]

print("Train:", train_data.shape)
print("Val:  ", val_data.shape)
print("Test: ", test_data.shape)

# Normalize using only train data statistics
scaler = StandardScaler()
scaler.fit(train_data)

train_scaled = scaler.transform(train_data)
val_scaled = scaler.transform(val_data)
test_scaled = scaler.transform(test_data)

train_scaled = train_scaled.astype(np.float32)
val_scaled = val_scaled.astype(np.float32)
test_scaled = test_scaled.astype(np.float32)

Train: (2211, 60)
Val:   (276, 60)
Test:  (277, 60)


In [4]:
# CELL 4 - Windowed dataset and dataloaders

class WindowedDataset(Dataset):
    def __init__(self, data, input_len=96, output_len=24):
        super().__init__()
        self.data = data
        self.input_len = input_len
        self.output_len = output_len
        self.total_len = len(data)

        self.max_start = self.total_len - (input_len + output_len)
        if self.max_start <= 0:
            raise ValueError("Not enough data for given input_len + output_len.")

    def __len__(self):
        return self.max_start

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.input_len]                   # (input_len, 60)
        y = self.data[idx + self.input_len:idx + self.input_len + self.output_len]  # (output_len, 60)
        return torch.from_numpy(x), torch.from_numpy(y)


input_len = 96
output_len = 24
num_features = train_scaled.shape[1]  # should be 60

train_ds = WindowedDataset(train_scaled, input_len=input_len, output_len=output_len)
val_ds = WindowedDataset(val_scaled, input_len=input_len, output_len=output_len)
test_ds = WindowedDataset(test_scaled, input_len=input_len, output_len=output_len)

batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

len(train_loader), len(val_loader), len(test_loader)

(65, 5, 5)

In [5]:
# CELL 5 - PatchTST model (compact implementation, in-notebook)

class PatchEmbedding(nn.Module):
    """
    Turn a multivariate time series into a sequence of patches.
    Input:  (B, C, L)
    Output: (B, N_patches, C * patch_len)
    """
    def __init__(self, patch_len: int, stride: int):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride

    def forward(self, x):
        # x: (B, C, L)
        B, C, L = x.shape
        x = x.unfold(dimension=2, size=self.patch_len, step=self.stride)  # (B, C, N_patches, patch_len)
        B, C, N, P = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()  # (B, N_patches, C, patch_len)
        x = x.view(B, N, C * P)  # (B, N_patches, C * patch_len)
        return x


class PositionalEncoding(nn.Module):
    """
    Standard sine-cosine positional encoding for patch indices.
    """
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) *
                             -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (B, N_patches, d_model)
        N = x.size(1)
        return x + self.pe[:, :N, :]


class PatchTSTEncoder(nn.Module):
    """
    Transformer encoder stack for patch representations.
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, num_layers: int, dropout: float = 0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        return self.encoder(x)


class PatchTST(nn.Module):
    """
    Compact PatchTST-style model (channel-independent-style over full feature dimension).

    Assumes input shape:  (B, L, C_total)
    """
    def __init__(
        self,
        input_len: int,
        output_len: int,
        num_channels: int,   # e.g. 60
        patch_len: int = 16,
        stride: int = 8,
        d_model: int = 128,
        n_heads: int = 8,
        d_ff: int = 256,
        num_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.input_len = input_len
        self.output_len = output_len
        self.num_channels = num_channels
        self.patch_len = patch_len
        self.stride = stride

        self.patch_embed = PatchEmbedding(patch_len=patch_len, stride=stride)

        # Project from (C * patch_len) to d_model
        self.proj = nn.Linear(num_channels * patch_len, d_model)

        # Positional encoding
        max_patches = (input_len - patch_len) // stride + 1
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=max_patches + 10)

        # Transformer encoder
        self.encoder = PatchTSTEncoder(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            num_layers=num_layers,
            dropout=dropout,
        )

        # Head: flatten patches then project to full forecast
        self.max_patches = max_patches
        self.head = nn.Sequential(
            nn.Flatten(start_dim=1),  # (B, N_patches * d_model)
            nn.Linear(max_patches * d_model, num_channels * output_len),
        )

    def forward(self, x):
        """
        x: (B, L, C_total)
        Returns: (B, output_len, C_total)
        """
        x = x.permute(0, 2, 1).contiguous()  # (B, C, L)
        x = self.patch_embed(x)              # (B, N_patches, C * patch_len)
        x = self.proj(x)                     # (B, N_patches, d_model)
        x = self.pos_enc(x)                  # (B, N_patches, d_model)
        x = self.encoder(x)                  # (B, N_patches, d_model)
        x = self.head(x)                     # (B, num_channels * output_len)
        x = x.view(x.size(0), self.output_len, self.num_channels)
        return x

In [9]:
# CELL 6 - Instantiate model, optimizer, LR scheduler, loss functions (tuned for 50 epochs)

model = PatchTST(
    input_len=input_len,
    output_len=output_len,
    num_channels=num_features,
    patch_len=8,        # smaller patches = better for noisy OHLCV
    stride=4,           # more overlap
    d_model=256,        # more capacity
    n_heads=8,
    d_ff=512,
    num_layers=3,
    dropout=0.1,
).to(device)

print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

base_lr = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=base_lr)
criterion_mse = nn.MSELoss()
criterion_mae = nn.L1Loss()

num_epochs = 50          # <--- UPDATED
warmup_epochs = 3        # warmup stays the same

def get_lr(epoch):
    """
    Cosine decay with warmup.
    epoch is 1-based.
    """
    if epoch <= warmup_epochs:
        return base_lr * epoch / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, (num_epochs - warmup_epochs))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


PatchTST(
  (patch_embed): PatchEmbedding()
  (proj): Linear(in_features=480, out_features=256, bias=True)
  (pos_enc): PositionalEncoding()
  (encoder): PatchTSTEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_featu

In [10]:
# CELL 7 - Training loop with warmup + cosine LR, progress bar, val MSE/MAE per epoch

for epoch in range(1, num_epochs + 1):
    # Update learning rate
    lr = get_lr(epoch)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    model.train()
    train_losses = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} - Training", leave=False)
    for xb, yb in pbar:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)

        loss = criterion_mse(preds, yb)
        loss.backward()

        # gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_losses.append(loss.item())
        pbar.set_postfix({"train_mse": f"{loss.item():.4f}", "lr": f"{lr:.2e}"})

    avg_train_loss = np.mean(train_losses)

    # Validation
    model.eval()
    val_mse_list = []
    val_mae_list = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            preds = model(xb)
            mse = criterion_mse(preds, yb).item()
            mae = criterion_mae(preds, yb).item()

            val_mse_list.append(mse)
            val_mae_list.append(mae)

    val_mse = np.mean(val_mse_list)
    val_mae = np.mean(val_mae_list)

    print(f"===== Epoch {epoch}/{num_epochs} =====")
    print(f"LR:       {lr:.6f}")
    print(f"Train MSE:{avg_train_loss:.6f}")
    print(f"Val   MSE:{val_mse:.6f}")
    print(f"Val   MAE:{val_mae:.6f}")

Epoch 1/50 - Training:   0%|          | 0/65 [00:00<?, ?it/s]

===== Epoch 1/50 =====
LR:       0.000333
Train MSE:0.289146
Val   MSE:1.791391
Val   MAE:1.015728


Epoch 2/50 - Training:   0%|          | 0/65 [00:00<?, ?it/s]

===== Epoch 2/50 =====
LR:       0.000667
Train MSE:0.164879
Val   MSE:1.933461
Val   MAE:1.045905


Epoch 3/50 - Training:   0%|          | 0/65 [00:00<?, ?it/s]

...

===== Epoch 48/50 =====
LR:       0.000004
Train MSE:0.006649
Val   MSE:1.590940
Val   MAE:0.920196


Epoch 49/50 - Training:   0%|          | 0/65 [00:00<?, ?it/s]

===== Epoch 49/50 =====
LR:       0.000001
Train MSE:0.006616
Val   MSE:1.594913
Val   MAE:0.921389


Epoch 50/50 - Training:   0%|          | 0/65 [00:00<?, ?it/s]

===== Epoch 50/50 =====
LR:       0.000000
Train MSE:0.006605
Val   MSE:1.594913
Val   MAE:0.921389


In [11]:
# CELL 8 - Final test evaluation

model.eval()
test_mse_list = []
test_mae_list = []

with torch.no_grad():
    for xb, yb in tqdm(test_loader, desc="Testing", leave=False):
        xb = xb.to(device)
        yb = yb.to(device)

        preds = model(xb)
        mse = criterion_mse(preds, yb).item()
        mae = criterion_mae(preds, yb).item()

        test_mse_list.append(mse)
        test_mae_list.append(mae)

test_mse = np.mean(test_mse_list)
test_mae = np.mean(test_mae_list)

print("===== FINAL TEST RESULTS (PatchTST) =====")
print(f"Test MSE: {test_mse:.6f}")
print(f"Test MAE: {test_mae:.6f}")

Testing:   0%|          | 0/5 [00:00<?, ?it/s]

===== FINAL TEST RESULTS (PatchTST) =====
Test MSE: 6.088604
Test MAE: 1.811146
